# PrimeVul Qwen2.5-Coder in-context baseline (Google Colab)

This is the other half of the midpoint baseline. The UniXcoder fine-tune is already done. This notebook does **not** train Qwen. It loads `Qwen/Qwen2.5-Coder-7B-Instruct` and asks it, separately, whether each function is vulnerable.

- **Data:** the original PrimeVul paired test file, 564 pairs, 1,128 functions. The vulnerable function and its patch are classified independently. The prompt does not say which one is which.
- **Task:** vulnerable vs benign. No CWE labels and no patch generation.
- **Conditions:** zero-shot, two-shot (the paper's two fixed examples), and chain-of-thought.
- **Metrics:** P-C, P-V, P-B, and P-R, same definitions as the UniXcoder notebook.

Use an A100 runtime (Runtime → Change runtime type → A100). Run the install cell, restart the runtime if Colab asks, then run the remaining cells in order. Read the smoke-test replies before the full run.

Results go to `MyDrive/primevul_qwen_icl/results` after every batch. If Colab disconnects, rerun the setup cells and the full-run cell. Finished functions are skipped.


In [ ]:
# Install, then restart the runtime if Colab says the kernel needs a restart.
# After a restart, continue from the next cell. You do not need to install again.
%pip install -q "transformers>=4.46,<5" "accelerate>=1,<2" "gdown>=5,<6"


## Setup

The model download is about 15 GB and stays on the Colab VM, not in Drive. The dataset is reused from `MyDrive/primevul_unixcoder_baseline/data` when that folder already has the original PrimeVul files. Otherwise this notebook downloads the same release the UniXcoder run used.

`RESUME = True` appends to existing prediction files. Set it to `False` only when you want to archive those files and start over. Changing the model, the prompts, or `MAX_INPUT_TOKENS` will not append to an old run.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os
import random
from pathlib import Path

# Inference only. Nothing in this notebook updates Qwen's weights.
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
SEED = 123456  # greedy decoding does not sample; the seed is recorded with the run
MAX_INPUT_TOKENS = 8192
MAX_NEW_TOKENS = {
    "zero_shot": 32,
    "two_shot": 32,
    "cot": 512,
}
BATCH_SIZE = {
    "zero_shot": 4,
    "two_shot": 4,
    "cot": 2,
}
# Edit this list to rerun one condition. Finished rows are skipped when RESUME is True.
RUN_CONDITIONS = ["zero_shot", "two_shot", "cot"]
RESUME = True

PROJECT_DIR = Path("/content/drive/MyDrive/primevul_qwen_icl")
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
LOCAL_HF_CACHE = Path("/content/hf_cache")
UNIXCODER_DATA_DIR = Path("/content/drive/MyDrive/primevul_unixcoder_baseline/data")

os.environ["HF_HOME"] = str(LOCAL_HF_CACHE)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

for path in (PROJECT_DIR, DATA_DIR, RESULTS_DIR, LOCAL_HF_CACHE):
    path.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
print("Project directory:", PROJECT_DIR)
print("Results directory:", RESULTS_DIR)
print("Conditions:", ", ".join(RUN_CONDITIONS))
print("Resume:", RESUME)


In [ ]:
import hashlib
import json
from pathlib import Path

PAIRED_NAME = "primevul_test_paired.jsonl"
# Same original-release folder the UniXcoder notebook uses. PrimeVul v0.1 is a different folder.
PRIMEVUL_FOLDER_URL = "https://drive.google.com/drive/folders/19iLaNDS0z99N8kB_jBRTmDLehwZBolMY"
N_PAIRS = 564
N_FUNCTIONS = N_PAIRS * 2


def locate_paired():
    matches = []
    for directory in (UNIXCODER_DATA_DIR, DATA_DIR):
        if directory.exists():
            matches.extend(directory.rglob(PAIRED_NAME))
    return sorted(matches, key=lambda path: len(path.parts))[0] if matches else None


PAIRED_PATH = locate_paired()
if PAIRED_PATH is None:
    import gdown

    print("Paired test file not found in Drive. Downloading the original PrimeVul release (about 450 MB)...")
    gdown.download_folder(
        url=PRIMEVUL_FOLDER_URL,
        output=str(DATA_DIR),
        quiet=False,
        use_cookies=False,
        remaining_ok=True,
    )
    PAIRED_PATH = locate_paired()
if PAIRED_PATH is None:
    raise RuntimeError(f"Download did not produce {PAIRED_NAME}")

raw_rows = []
with PAIRED_PATH.open(encoding="utf-8") as handle:
    for line in handle:
        if line.strip():
            raw_rows.append(json.loads(line))
if len(raw_rows) != N_FUNCTIONS:
    raise RuntimeError(
        f"{PAIRED_PATH} has {len(raw_rows)} rows. "
        f"The midpoint baseline used the original paired test file, which has {N_FUNCTIONS} rows ({N_PAIRS} pairs)."
    )

ROWS = []
for index, row in enumerate(raw_rows):
    if "func" not in row or "target" not in row:
        raise RuntimeError(f"Row {index} is missing func or target")
    func = row["func"]
    if not isinstance(func, str) or not func.strip():
        raise RuntimeError(f"Row {index} has an empty function")
    target = int(row["target"])
    if target not in (0, 1):
        raise RuntimeError(f"Row {index} has target {row['target']!r}")
    item = {
        "row_index": index,
        "pair_number": index // 2,
        "target": target,
        "func": func,
        "idx": row.get("idx"),
    }
    for key in ("project", "commit_id"):
        if isinstance(row.get(key), (str, int)) and not isinstance(row.get(key), bool):
            item[key] = row[key]
    ROWS.append(item)

for pair_number in range(N_PAIRS):
    targets = (ROWS[2 * pair_number]["target"], ROWS[2 * pair_number + 1]["target"])
    if sum(targets) != 1:
        raise RuntimeError(f"Pair {pair_number} targets are {targets}. Expected one vulnerable and one benign function.")

digest = hashlib.sha256(PAIRED_PATH.read_bytes()).hexdigest()
PAIRED_SHA256 = digest
PAIRED_BYTES = PAIRED_PATH.stat().st_size
vulnerable_first = sum(row["target"] == 1 and row["row_index"] % 2 == 0 for row in ROWS)
print(f"Using {PAIRED_PATH}")
print(f"sha256 {PAIRED_SHA256}  bytes {PAIRED_BYTES}")
print(f"columns in the first row: {sorted(raw_rows[0])}")
print(
    f"test_paired: {len(ROWS)} rows | vulnerable={sum(row['target'] == 1 for row in ROWS)} | "
    f"benign={sum(row['target'] == 0 for row in ROWS)} | pairs={N_PAIRS}"
)
print(f"vulnerable function is the first member in {vulnerable_first}/{N_PAIRS} pairs")


## Prompts and pair metrics

The prompt text is the PrimeVul OpenAI setup, copied from `openai_expr/utils.py`:

1. **zero-shot** is `std_cls` with no examples.
2. **two-shot** is `std_cls` plus the paper's two fixed examples (one labeled NO, one labeled YES). Those examples are not the test function's patch.
3. **chain-of-thought** is `cot`, with no examples, ending in "Let's think step-by-step."

A reply of YES means vulnerable and NO means benign. Pair outcomes use the same rules as the UniXcoder notebook:

- **P-C:** vulnerable detected and patch labeled benign
- **P-V:** both labeled vulnerable
- **P-B:** both labeled benign
- **P-R:** vulnerable labeled benign and patch labeled vulnerable

A reply that does not contain a verdict does not become a benign prediction. That pair is counted as unparsed. The checks at the bottom of the next cell fail the cell if the parser or the pair rules drift.


In [ ]:
# Prompts copied from the PrimeVul OpenAI scripts.
# zero_shot = std_cls, two_shot = std_cls with --fewshot_eg, cot = cot without examples.
# Source: https://github.com/DLVulDet/PrimeVul/blob/main/openai_expr/utils.py

UNIXCODER_REFERENCE_COUNTS = {"P-C": 8, "P-V": 76, "P-B": 476, "P-R": 4}

# **Standard Classification Prompt**

SYS_INST = "You are a security expert that is good at static program analysis."

PROMPT_INST = """Please analyze the following code:
```
{func}
```
Please indicate your analysis result with one of the options: 
(1) YES: A security vulnerability detected.
(2) NO: No security vulnerability. 

Only reply with one of the options above. Do not include any further information.
"""


# **Chain of Thought Prompt**

PROMPT_INST_COT = """Please analyze the following code:
```
{func}
```
Please indicate your analysis result with one of the options: 
(1) YES: A security vulnerability detected.
(2) NO: No security vulnerability. 

Make sure to include one of the options above "explicitly" (EXPLICITLY!!!) in your response.
Let's think step-by-step.
"""

ONESHOT_USER = """Please analyze the following code:
```
static char *clean_path(char *path)
{
        char *ch;
        char *ch2;
        char *str;
        str = xmalloc(strlen(path) + 1);
        ch = path;
        ch2 = str;
        while (true) {
                *ch2 = *ch;
                ch++;
                ch2++;
                if (!*(ch-1))
                        break;
                while (*(ch - 1) == '/' && *ch == '/')
                        ch++;
        }
        /* get rid of trailing / characters */
        while ((ch = strrchr(str, '/'))) {
                if (ch == str)
                        break;
                if (!*(ch+1))
                        *ch = 0;
                else
                        break;
        }
        return str;
}
```
Please indicate your analysis result with one of the options: 
(1) YES: A security vulnerability detected.
(2) NO: No security vulnerability. 

Only reply with one of the options above. Do not include any further information.
"""

ONESHOT_ASSISTANT = "NO"

TWOSHOT_USER = """Please analyze the following code:
```
int64 ClientUsageTracker::GetCachedHostUsage(const std::string& host) {
   HostUsageMap::const_iterator found = cached_usage_.find(host);
   if (found == cached_usage_.end())
     return 0;

  int64 usage = 0;
  const UsageMap& map = found->second;
  for (UsageMap::const_iterator iter = map.begin();
       iter != map.end(); ++iter) {
    usage += iter->second;
  }
  return usage;
}

```
Please indicate your analysis result with one of the options: 
(1) YES: A security vulnerability detected.
(2) NO: No security vulnerability. 

Only reply with one of the options above. Do not include any further information.
"""

TWOSHOT_ASSISTANT = "YES"

import csv
import json
import re

OUTCOMES = ("P-C", "P-V", "P-B", "P-R")
N_PAIRS = 564
N_FUNCTIONS = N_PAIRS * 2

# A copied instruction lists both options within a short span. Using the last of
# those two would mark every echoed menu as benign, because option (2) is NO.
# CoT replies often state the option first and then keep analyzing, so the option
# text counts wherever it appears. A bare "no" inside the analysis does not.
_VERDICT = re.compile(
    r"\(\s*[12]\s*\)\s*(?P<paren>YES|Yes|NO|No)\b"
    r"|EXPLICITLY!!!\s*(?P<explicit>YES|Yes|NO|No)\b"
    r"|(?P<yes_option>YES|Yes)\s*:\s*A security vulnerability detected"
    r"|(?P<no_option>NO|No)\s*:\s*No security vulnerability"
)
_MENU_GAP = 300


def _match_label(match):
    for name in ("paren", "explicit", "yes_option", "no_option"):
        word = match.group(name)
        if word:
            return word
    raise RuntimeError("Verdict pattern matched without a label")


def fill_func(template, func):
    marker = "{func}"
    if template.count(marker) != 1:
        raise ValueError("Prompt template must contain {func} exactly once")
    return template.replace(marker, func)


def messages_for(condition, func):
    """Build one chat. The label of this function is not included."""
    if condition == "cot":
        template = PROMPT_INST_COT
    elif condition in ("zero_shot", "two_shot"):
        template = PROMPT_INST
    else:
        raise ValueError(f"Unknown condition: {condition}")
    messages = [{"role": "system", "content": SYS_INST}]
    if condition == "two_shot":
        messages.extend([
            {"role": "user", "content": ONESHOT_USER},
            {"role": "assistant", "content": ONESHOT_ASSISTANT},
            {"role": "user", "content": TWOSHOT_USER},
            {"role": "assistant", "content": TWOSHOT_ASSISTANT},
        ])
    messages.append({"role": "user", "content": fill_func(template, func)})
    return messages


def _word_verdict(word):
    return 1 if word.upper() == "YES" else 0


def _short_line_verdict(line):
    """Accept a short final line that is the verdict, not a sentence containing 'no'."""
    raw = line.strip().strip("`*_ ").strip().rstrip(".").strip()
    if not raw or len(raw) > 180:
        return None
    compact = re.sub(r"\s+", " ", raw)
    lowered = compact.lower()
    exact = {
        "yes": 1,
        "no": 0,
        "(1)": 1,
        "(2)": 0,
        "(1) yes": 1,
        "(2) no": 0,
    }
    if lowered in exact:
        return exact[lowered]
    if re.fullmatch(r"(?:\(\s*1\s*\)\s*)?(?:YES|Yes)\b(?:\s*[:\-].*)?", compact):
        return 1
    if re.fullmatch(r"(?:\(\s*2\s*\)\s*)?(?:NO|No)\b(?:\s*[:\-].*)?", compact):
        return 0
    if len(lowered) <= 80 and re.fullmatch(r"(?:\(\s*1\s*\)\s*)?yes\b(?:\s*[:\-].*)?", lowered):
        return 1
    if len(lowered) <= 80 and re.fullmatch(r"(?:\(\s*2\s*\)\s*)?no\b(?:\s*[:\-].*)?", lowered):
        return 0
    match = re.search(r"\b(YES|Yes|NO|No)\b\s*$", compact)
    if match and len(compact) <= 80:
        return _word_verdict(match.group(1))
    if len(lowered) <= 80 and re.search(r"\b(answer|verdict|result)\b", lowered):
        tail = re.search(r"\b(yes|no)\b\s*$", lowered)
        if tail:
            return 1 if tail.group(1) == "yes" else 0
    return None


def parse_verdict(text):
    """Return 1 for vulnerable, 0 for benign, or None when the reply has no verdict.

    Chain-of-thought replies often say "no bounds check". Searching the whole
    reply for the word "no" would score those as benign. A reply counts when it
    uses the option text, including when the option comes first and the analysis
    follows. If both options appear close together and nothing follows, that is
    a copied menu, not a decision, so the reply stays unparsed.
    """
    if not isinstance(text, str):
        return None
    stripped = text.strip()
    if not stripped:
        return None
    matches = list(_VERDICT.finditer(stripped))
    if len(matches) == 1:
        return _word_verdict(_match_label(matches[0]))
    if len(matches) > 1:
        last = matches[-1]
        last_label = _match_label(last).upper()
        opposite = next(
            (
                match for match in reversed(matches[:-1])
                if _match_label(match).upper() != last_label
            ),
            None,
        )
        if opposite is not None and last.start() - opposite.end() < _MENU_GAP:
            return parse_verdict(stripped[last.end():])
        return _word_verdict(last_label)
    lines = [line.strip() for line in stripped.splitlines() if line.strip()]
    if not lines:
        return None
    return _short_line_verdict(lines[-1])


def prediction_path(condition):
    return RESULTS_DIR / f"{condition}.jsonl"


def load_predictions(path):
    by_index = {}
    bad_lines = 0
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                bad_lines += 1
                continue
            by_index[int(row["row_index"])] = row
    if bad_lines:
        print(f"Ignored {bad_lines} incomplete line(s) in {path.name}")
    return by_index


def function_scores(records):
    tp = fp = tn = fn = unparsed = 0
    for row in records:
        target = int(row["target"])
        prediction = row.get("prediction")
        if prediction not in (0, 1):
            unparsed += 1
            continue
        if prediction == 1 and target == 1:
            tp += 1
        elif prediction == 1 and target == 0:
            fp += 1
        elif prediction == 0 and target == 0:
            tn += 1
        else:
            fn += 1
    parsed = tp + fp + tn + fn
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    accuracy = (tp + tn) / parsed if parsed else 0.0
    return {
        "parsed": parsed,
        "unparsed": unparsed,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "accuracy_pct": 100.0 * accuracy,
        "precision_pct": 100.0 * precision,
        "recall_pct": 100.0 * recall,
        "f1_pct": 100.0 * f1,
        "predicted_vulnerable": tp + fp,
    }


def pair_report(records, expected_pairs=N_PAIRS):
    """Score adjacent file rows. P-C/P-V/P-B/P-R match the UniXcoder notebook."""
    by_index = {}
    for row in records:
        index = int(row["row_index"])
        if not 0 <= index < expected_pairs * 2:
            raise RuntimeError(f"row_index {index} is outside the paired file")
        by_index[index] = row
    counts = {name: 0 for name in (*OUTCOMES, "unparsed")}
    key = {(1, 0): "P-C", (1, 1): "P-V", (0, 0): "P-B", (0, 1): "P-R"}
    for pair_number in range(expected_pairs):
        left = by_index.get(2 * pair_number)
        right = by_index.get(2 * pair_number + 1)
        if left is None or right is None:
            continue
        members = {}
        for item in (left, right):
            target = int(item["target"])
            if target in members:
                raise RuntimeError(f"Pair {pair_number} does not have one of each label")
            members[target] = item.get("prediction")
        if set(members) != {0, 1}:
            raise RuntimeError(f"Pair {pair_number} labels are {sorted(members)}")
        vulnerable = members[1]
        benign = members[0]
        if vulnerable in (0, 1) and benign in (0, 1):
            counts[key[(vulnerable, benign)]] += 1
        else:
            counts["unparsed"] += 1
    scored = sum(counts.values())
    return {
        "counts": counts,
        "scored_pairs": scored,
        "missing_pairs": expected_pairs - scored,
        "complete": scored == expected_pairs and len(by_index) == expected_pairs * 2,
    }


def _percent(count, total):
    return 100.0 * count / total if total else 0.0


def _saved_or_global(config, key, name):
    if config.get(key) is not None:
        return config[key]
    if name in globals():
        return globals()[name]
    raise RuntimeError(f"Run the setup cell first. Missing {name}.")


def summarize():
    """Read saved JSONL files and write metrics.json plus summary.csv."""
    config = {}
    config_path = RESULTS_DIR / "run_config.json"
    if config_path.exists():
        config = json.loads(config_path.read_text(encoding="utf-8"))
    elapsed = {}
    elapsed_path = RESULTS_DIR / "elapsed.json"
    if elapsed_path.exists():
        elapsed = json.loads(elapsed_path.read_text(encoding="utf-8"))
    max_new_tokens = _saved_or_global(config, "max_new_tokens", "MAX_NEW_TOKENS")

    payload = {
        "model": _saved_or_global(config, "model", "MODEL_NAME"),
        "task": "binary vulnerability detection, inference only",
        "split": "primevul_test_paired",
        "functions": N_FUNCTIONS,
        "pairs": N_PAIRS,
        "decoding": "greedy",
        "max_input_tokens": _saved_or_global(config, "max_input_tokens", "MAX_INPUT_TOKENS"),
        "dataset_sha256": config.get("dataset_sha256"),
        "gpu": config.get("gpu"),
        "conditions": {},
        "reference_not_from_this_run": {
            "model": "microsoft/unixcoder-base",
            "source": "colab/midpoint_baseline.json",
            "paired_counts": dict(UNIXCODER_REFERENCE_COUNTS),
        },
    }
    csv_rows = []
    found = False
    print(f"{'condition':16s}  " + "  ".join(f"{name:>16s}" for name in (*OUTCOMES, "unparsed")))
    reference = payload["reference_not_from_this_run"]["paired_counts"]
    ref_parts = [f"{'unixcoder':16s}"]
    for name in OUTCOMES:
        ref_parts.append(f"{reference[name]:4d} ({_percent(reference[name], N_PAIRS):6.2f}%)")
    ref_parts.append(f"{0:4d} ({0:6.2f}%)")
    for condition in ("zero_shot", "two_shot", "cot"):
        path = prediction_path(condition)
        if not path.exists():
            print(f"{condition:16s}  no saved predictions")
            continue
        found = True
        records = list(load_predictions(path).values())
        report = pair_report(records)
        scores = function_scores(records)
        counts = report["counts"]
        parts = [f"{condition:16s}"]
        for name in (*OUTCOMES, "unparsed"):
            if report["complete"]:
                parts.append(f"{counts[name]:4d} ({_percent(counts[name], N_PAIRS):6.2f}%)")
            else:
                parts.append(f"{counts[name]:4d}")
        print("  ".join(parts))
        if not report["complete"]:
            print(
                f"  INCOMPLETE: {len(records)}/{N_FUNCTIONS} functions, "
                f"{report['missing_pairs']} pairs still missing. These are not the check-in numbers yet."
            )
        elif counts["unparsed"]:
            print("  Some replies had no YES/NO verdict. Those pairs are counted as unparsed, not as P-B.")
        print(
            f"  function-level on parsed replies: "
            f"F1 {scores['f1_pct']:.2f}%  precision {scores['precision_pct']:.2f}%  "
            f"recall {scores['recall_pct']:.2f}%  "
            f"predicted vulnerable {scores['predicted_vulnerable']}/{scores['parsed']}"
        )
        if scores["parsed"] and (
            scores["predicted_vulnerable"] <= 0.01 * scores["parsed"]
            or scores["predicted_vulnerable"] >= 0.99 * scores["parsed"]
        ):
            print("  WARNING: almost every parsed function got the same label. Check the raw replies before reporting.")
        truncated = sum(bool(row.get("truncated")) for row in records)
        if truncated:
            print(f"  truncated functions: {truncated}/{len(records)}")
        condition_payload = {
            "max_new_tokens": max_new_tokens[condition],
            "complete": report["complete"],
            "functions_saved": len(records),
            "scored_pairs": report["scored_pairs"],
            "missing_pairs": report["missing_pairs"],
            "counts": counts,
            "percent_of_564_pairs": (
                {name: _percent(counts[name], N_PAIRS) for name in counts} if report["complete"] else None
            ),
            "truncated_functions": truncated,
            "elapsed_seconds": elapsed.get(condition),
            "function_level": scores,
        }
        payload["conditions"][condition] = condition_payload
        csv_rows.append({
            "condition": condition,
            "complete": report["complete"],
            "P-C": counts["P-C"],
            "P-V": counts["P-V"],
            "P-B": counts["P-B"],
            "P-R": counts["P-R"],
            "unparsed": counts["unparsed"],
            "function_f1_pct": round(scores["f1_pct"], 4),
            "predicted_vulnerable": scores["predicted_vulnerable"],
            "parsed_functions": scores["parsed"],
            "truncated_functions": truncated,
            "elapsed_seconds": elapsed.get(condition),
        })
    print("  ".join(ref_parts))
    print("unixcoder is the finished fine-tune, printed here only as a reference. It is not a Qwen result.")
    if found:
        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        (RESULTS_DIR / "metrics.json").write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
        with (RESULTS_DIR / "summary.csv").open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(csv_rows[0]))
            writer.writeheader()
            writer.writerows(csv_rows)
        print(f"Wrote {RESULTS_DIR / 'metrics.json'}")
        print(f"Wrote {RESULTS_DIR / 'summary.csv'}")
    else:
        print("No Qwen prediction files yet.")
    return payload


def _check_prompts_and_parser():
    assert ONESHOT_ASSISTANT == "NO"
    assert TWOSHOT_ASSISTANT == "YES"
    assert UNIXCODER_REFERENCE_COUNTS == {"P-C": 8, "P-V": 76, "P-B": 476, "P-R": 4}
    assert sum(UNIXCODER_REFERENCE_COUNTS.values()) == N_PAIRS
    sample = "int main() { return {0}; }"
    assert sample in fill_func(PROMPT_INST, sample)
    assert "{func}" not in fill_func(PROMPT_INST, sample)
    zero = messages_for("zero_shot", sample)
    two = messages_for("two_shot", sample)
    cot = messages_for("cot", sample)
    assert [item["role"] for item in zero] == ["system", "user"]
    assert [item["role"] for item in two] == ["system", "user", "assistant", "user", "assistant", "user"]
    assert [item["role"] for item in cot] == ["system", "user"]
    assert two[2]["content"] == "NO" and two[4]["content"] == "YES"
    assert "clean_path" in two[1]["content"] and "GetCachedHostUsage" in two[3]["content"]
    assert sample in two[-1]["content"] and sample in cot[-1]["content"]
    assert "step-by-step" in cot[-1]["content"]
    assert "step-by-step" not in zero[-1]["content"]
    assert "target" not in zero[-1]["content"]

    assert parse_verdict("YES") == 1
    assert parse_verdict("NO") == 0
    assert parse_verdict("yes") == 1
    assert parse_verdict("(1) YES: A security vulnerability detected.") == 1
    assert parse_verdict("(2) NO: No security vulnerability.") == 0
    assert parse_verdict("There is no bounds check.\n(1) YES") == 1
    assert parse_verdict("There is no bounds check.") is None
    assert parse_verdict(
        "EXPLICITLY!!! NO: No security vulnerability. ### Analysis: The copy has no bounds check."
    ) == 0
    assert parse_verdict(
        "EXPLICITLY!!! YES: A security vulnerability detected. The buffer can overflow."
    ) == 1
    assert parse_verdict(
        "Let's analyze the code step-by-step. There is no bounds check on the copy."
    ) is None
    assert parse_verdict("Final answer: NO") == 0
    assert parse_verdict("Final answer: no") == 0
    assert parse_verdict("I considered YES, but the final answer is NO") == 0
    assert parse_verdict(None) is None
    assert parse_verdict("   ") is None
    menu = (
        "(1) YES: A security vulnerability detected.\n"
        "(2) NO: No security vulnerability.\n"
        "YES"
    )
    assert parse_verdict(menu) == 1
    assert parse_verdict(
        "(1) YES: A security vulnerability detected.\n(2) NO: No security vulnerability."
    ) is None
    distant = "(1) YES\n\n" + ("The copy walks off the end of the buffer.\n" * 30) + "(2) NO"
    assert parse_verdict(distant) == 0

    records = []
    samples = [
        (1, 1, 0, 0),
        (1, 1, 0, 1),
        (0, 0, 1, 0),
        (1, 0, 0, 1),
        (1, None, 0, 0),
    ]
    for pair_number, (target_a, pred_a, target_b, pred_b) in enumerate(samples):
        records.append({"row_index": 2 * pair_number, "target": target_a, "prediction": pred_a})
        records.append({"row_index": 2 * pair_number + 1, "target": target_b, "prediction": pred_b})
    report = pair_report(records, expected_pairs=5)
    assert report["counts"] == {"P-C": 1, "P-V": 1, "P-B": 1, "P-R": 1, "unparsed": 1}
    scores = function_scores([
        {"target": 1, "prediction": 1},
        {"target": 0, "prediction": 0},
        {"target": 0, "prediction": 1},
        {"target": 1, "prediction": None},
    ])
    assert (scores["tp"], scores["fp"], scores["tn"], scores["fn"], scores["unparsed"]) == (1, 1, 1, 0, 1)
    print("Prompt, parser, and pair-metric checks passed.")


_check_prompts_and_parser()


## Load the model

BF16 on one GPU. The first run downloads the weights. Later runs in the same VM reuse `/content/hf_cache`. A disconnect deletes that cache with the VM, so the next session downloads it again.


In [ ]:
import random

import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. In Colab, choose Runtime, Change runtime type, then A100.")

gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name} ({gpu_gb:.1f} GB)")
print(f"PyTorch: {torch.__version__}")
if gpu_gb < 30:
    print(
        "WARNING: Qwen2.5-Coder-7B in BF16 is set up for an A100 (40 GB or 80 GB). "
        "This GPU is smaller. Loading may run out of memory."
    )

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=str(LOCAL_HF_CACHE))
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

load_kwargs = {
    "torch_dtype": torch.bfloat16,
    "device_map": {"": 0},
    "cache_dir": str(LOCAL_HF_CACHE),
}
try:
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, attn_implementation="sdpa", **load_kwargs)
    except (ValueError, ImportError):
        print("SDPA attention is unavailable. Loading the default attention implementation.")
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
except torch.cuda.OutOfMemoryError as exc:
    raise RuntimeError(
        "The model does not fit on this GPU in BF16. Change the Colab runtime to an A100 and rerun."
    ) from exc
model.eval()
# The checkpoint's own config samples (temperature 0.7, repetition penalty 1.05).
# Override both so every condition is greedy, matching PrimeVul's temperature 0.
model.generation_config.do_sample = False
model.generation_config.repetition_penalty = 1.0
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None
for parameter in model.parameters():
    parameter.requires_grad_(False)

context = int(model.config.max_position_embeddings)
tokenizer.model_max_length = context
needed = MAX_INPUT_TOKENS + max(MAX_NEW_TOKENS.values())
if needed > context:
    raise RuntimeError(f"Input plus generation ({needed} tokens) exceeds the model context ({context}).")
allocated = torch.cuda.memory_allocated() / 1024**3
print(f"Loaded {MODEL_NAME} in BF16. Context {context} tokens. GPU memory allocated {allocated:.1f} GB.")
print(f"Pad token: {tokenizer.pad_token!r}   EOS: {tokenizer.eos_token!r}")
print("Weights are frozen. This run does not train.")


## Generate

The smoke test classifies the first 2 pairs in each selected condition and prints the raw replies. It does not write results. Each reply should show `-> 1` or `-> 0`. `-> None` means the parser did not find a verdict.

The full-run cell writes one JSONL per condition. Zero-shot and two-shot stop at 32 new tokens. Chain-of-thought stops at 512. Decoding is greedy, which is the temperature-0 setting from the PrimeVul prompting scripts. Batching only pads independent prompts; a function never sees its pair partner.


In [ ]:
import gc
import hashlib
import json
import os
import time

import torch
import transformers

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable


def encode_messages(messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"]


def prepare_row(condition, row):
    """Tokenize one function. Truncate the function body, never the YES/NO instruction."""
    func = row["func"]
    truncated = False
    # A character cap keeps one huge function from dominating tokenization time.
    # 16 characters per token is above real C/C++ code, so this does not cut a
    # function that would have fit in MAX_INPUT_TOKENS.
    char_cap = MAX_INPUT_TOKENS * 16
    if len(func) > char_cap:
        func = func[:char_cap]
        truncated = True
    ids = encode_messages(messages_for(condition, func))
    if len(ids) > MAX_INPUT_TOKENS:
        truncated = True
        func_ids = tokenizer(func, add_special_tokens=False, truncation=False)["input_ids"]
        empty_len = len(encode_messages(messages_for(condition, "")))
        keep = MAX_INPUT_TOKENS - empty_len
        if keep < 1:
            raise RuntimeError(
                f"The {condition} prompt does not fit in MAX_INPUT_TOKENS={MAX_INPUT_TOKENS} "
                "even with an empty function."
            )
        for _ in range(16):
            keep = max(0, min(keep, len(func_ids)))
            trimmed = tokenizer.decode(func_ids[:keep], clean_up_tokenization_spaces=False) if keep else ""
            ids = encode_messages(messages_for(condition, trimmed))
            if len(ids) <= MAX_INPUT_TOKENS:
                break
            keep -= max(len(ids) - MAX_INPUT_TOKENS, 1) + 8
        if len(ids) > MAX_INPUT_TOKENS:
            raise RuntimeError(
                f"Row {row['row_index']} is still {len(ids)} tokens after truncation to {MAX_INPUT_TOKENS}."
            )
    rendered = tokenizer.decode(ids, clean_up_tokenization_spaces=False)
    marker = "Let's think step-by-step." if condition == "cot" else "Only reply with one of the options above."
    if marker not in rendered:
        raise RuntimeError(
            f"The {condition} prompt for row {row['row_index']} lost its answer instruction. "
            "Stop and keep this notebook; do not report numbers from a prompt that dropped the question."
        )
    prepared = {
        "row_index": row["row_index"],
        "pair_number": row["pair_number"],
        "idx": row["idx"],
        "target": row["target"],
        "input_ids": ids,
        "truncated": truncated,
        "condition": condition,
    }
    for key in ("project", "commit_id"):
        if key in row:
            prepared[key] = row[key]
    return prepared


def _is_oom(exc):
    oom_type = getattr(getattr(torch, "cuda", None), "OutOfMemoryError", ())
    if oom_type and isinstance(exc, oom_type):
        return True
    return "out of memory" in str(exc).lower()


def generate_texts(input_id_lists, max_new_tokens):
    width = max(len(ids) for ids in input_id_lists)
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        raise RuntimeError("The tokenizer has no pad token.")
    batch_ids = []
    batch_mask = []
    for ids in input_id_lists:
        pad = width - len(ids)
        batch_ids.append([pad_id] * pad + list(ids))
        batch_mask.append([0] * pad + [1] * len(ids))
    device = next(model.parameters()).device
    input_ids = torch.tensor(batch_ids, dtype=torch.long, device=device)
    attention_mask = torch.tensor(batch_mask, dtype=torch.long, device=device)
    with torch.inference_mode():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.0,
            pad_token_id=pad_id,
            eos_token_id=model.generation_config.eos_token_id,
            use_cache=True,
        )
    if output.shape[1] < width:
        raise RuntimeError("The model returned fewer tokens than the prompt.")
    return tokenizer.batch_decode(output[:, width:].detach().cpu(), skip_special_tokens=True)


def generate_resilient(items, max_new_tokens):
    try:
        return generate_texts([item["input_ids"] for item in items], max_new_tokens)
    except Exception as exc:
        if not _is_oom(exc):
            raise
        torch.cuda.empty_cache()
        if len(items) == 1:
            raise RuntimeError(
                "Out of memory on a single function. Lower MAX_INPUT_TOKENS and use a new PROJECT_DIR."
            ) from exc
        mid = max(1, len(items) // 2)
        print(f"Out of memory at batch {len(items)}. Retrying as {mid} + {len(items) - mid}.")
        left = generate_resilient(items[:mid], max_new_tokens)
        right = generate_resilient(items[mid:], max_new_tokens)
        return left + right


def config_fingerprint():
    payload = {
        "model": MODEL_NAME,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "decoding": "greedy",
        "dataset_sha256": PAIRED_SHA256,
        "sys": SYS_INST,
        "prompt": PROMPT_INST,
        "prompt_cot": PROMPT_INST_COT,
        "oneshot_user": ONESHOT_USER,
        "oneshot_assistant": ONESHOT_ASSISTANT,
        "twoshot_user": TWOSHOT_USER,
        "twoshot_assistant": TWOSHOT_ASSISTANT,
    }
    raw = json.dumps(payload, sort_keys=True).encode()
    return hashlib.sha256(raw).hexdigest()


def prepare_run_files():
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    current = {
        "fingerprint": config_fingerprint(),
        "model": MODEL_NAME,
        "gpu": torch.cuda.get_device_name(0),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "batch_size": BATCH_SIZE,
        "decoding": "greedy",
        "seed": SEED,
        "dataset_path": str(PAIRED_PATH),
        "dataset_sha256": PAIRED_SHA256,
        "dataset_bytes": PAIRED_BYTES,
        "pairs": N_PAIRS,
        "notes": (
            "Each function is classified independently. "
            "The two-shot examples are the paper's fixed examples, not test-pair partners."
        ),
    }
    path = RESULTS_DIR / "run_config.json"
    saved_names = [name for name in ("zero_shot", "two_shot", "cot") if prediction_path(name).exists()]
    has_outputs = bool(saved_names) or (RESULTS_DIR / "elapsed.json").exists() or (RESULTS_DIR / "metrics.json").exists()
    if path.exists():
        saved = json.loads(path.read_text(encoding="utf-8"))
        mismatch = saved.get("fingerprint") != current["fingerprint"]
    else:
        mismatch = bool(saved_names)
    if mismatch and RESUME:
        raise RuntimeError(
            "Saved predictions were produced with a different model, prompt, token cap, or dataset file, "
            "or prediction files exist without run_config.json. "
            "Set PROJECT_DIR to a new folder, or set RESUME = False to archive the old files and start over."
        )
    if has_outputs and (mismatch or not RESUME):
        _archive_results()
    path.write_text(json.dumps(current, indent=2) + "\n", encoding="utf-8")
    return current


def _archive_results():
    stamp = time.strftime("%Y%m%d-%H%M%S")
    for name in ("zero_shot.jsonl", "two_shot.jsonl", "cot.jsonl", "metrics.json", "summary.csv", "elapsed.json"):
        path = RESULTS_DIR / name
        if path.exists():
            backup = path.with_name(f"{path.name}.bak-{stamp}")
            path.rename(backup)
            print(f"Archived {path.name} to {backup.name}")


def _record(item, text):
    record = {
        "row_index": item["row_index"],
        "pair_number": item["pair_number"],
        "idx": item["idx"],
        "target": item["target"],
        "prediction": parse_verdict(text),
        "response": text,
        "prompt_tokens": len(item["input_ids"]),
        "truncated": item["truncated"],
        "condition": item["condition"],
    }
    for key in ("project", "commit_id"):
        if key in item:
            record[key] = item[key]
    return record


def evaluate_condition(condition):
    if condition not in MAX_NEW_TOKENS:
        raise RuntimeError(f"Unknown condition: {condition}")
    path = prediction_path(condition)
    done = load_predictions(path) if path.exists() else {}
    pending = [row for row in ROWS if row["row_index"] not in done]
    print(f"\n{condition}: {len(done)} already saved, {len(pending)} left")
    if not pending:
        return
    started = time.time()
    prepared = []
    for row in tqdm(pending, desc=f"{condition} tokenize"):
        prepared.append(prepare_row(condition, row))
    lengths = [len(item["input_ids"]) for item in prepared]
    truncated = sum(item["truncated"] for item in prepared)
    print(
        f"{condition}: prompt tokens min {min(lengths)}  mean {sum(lengths) / len(lengths):.0f}  "
        f"max {max(lengths)}  truncated {truncated}/{len(prepared)}"
    )
    if truncated:
        print(
            f"WARNING: {truncated} functions were cut so the prompt fits in {MAX_INPUT_TOKENS} tokens. "
            "The YES/NO instruction is kept. Raise MAX_INPUT_TOKENS to send more of each function, "
            "then set RESUME = False so the old replies are not mixed in."
        )
    ordered = sorted(prepared, key=lambda item: len(item["input_ids"]))
    batch_size = BATCH_SIZE[condition]
    max_new = MAX_NEW_TOKENS[condition]
    with path.open("a", encoding="utf-8") as handle:
        batches = range(0, len(ordered), batch_size)
        for start in tqdm(batches, desc=f"{condition} generate"):
            batch = ordered[start:start + batch_size]
            texts = generate_resilient(batch, max_new)
            if len(texts) != len(batch):
                raise RuntimeError("The model returned a different number of replies than the batch.")
            for item, text in zip(batch, texts):
                record = _record(item, text)
                try:
                    line = json.dumps(record, ensure_ascii=False)
                except UnicodeEncodeError:
                    record["response"] = record["response"].encode("utf-8", "replace").decode("utf-8")
                    line = json.dumps(record, ensure_ascii=False)
                handle.write(line + "\n")
            handle.flush()
            try:
                os.fsync(handle.fileno())
            except OSError:
                pass
    elapsed = time.time() - started
    elapsed_path = RESULTS_DIR / "elapsed.json"
    elapsed_state = {}
    if elapsed_path.exists():
        elapsed_state = json.loads(elapsed_path.read_text(encoding="utf-8"))
    elapsed_state[condition] = round(elapsed_state.get(condition, 0) + elapsed, 1)
    elapsed_path.write_text(json.dumps(elapsed_state, indent=2) + "\n", encoding="utf-8")
    print(f"{condition} finished in {elapsed / 60:.1f} min this session.")


def warn_if_test_rows_match_examples():
    """The two-shot prompt already contains the label for these two fixed functions."""
    def body(message):
        start = message.index("```\n") + 4
        end = message.index("\n```", start)
        return message[start:end].strip()

    examples = {body(ONESHOT_USER), body(TWOSHOT_USER)}
    hits = [row["row_index"] for row in ROWS if row["func"].strip() in examples]
    if hits:
        print(
            "WARNING: these test rows are the same code as a two-shot example, so that prompt contains the answer: "
            + ", ".join(str(index) for index in hits)
        )


def smoke_test(n_pairs=2):
    """Run a few pairs in every selected condition. Nothing is written to Drive."""
    warn_if_test_rows_match_examples()
    print(f"Smoke test on the first {n_pairs} pairs ({n_pairs * 2} functions). These replies are not saved.")
    for condition in RUN_CONDITIONS:
        started = time.time()
        prepared = [prepare_row(condition, ROWS[index]) for index in range(n_pairs * 2)]
        texts = []
        batch_size = BATCH_SIZE[condition]
        for start in range(0, len(prepared), batch_size):
            texts.extend(generate_resilient(prepared[start:start + batch_size], MAX_NEW_TOKENS[condition]))
        elapsed = time.time() - started
        print(f"\n=== {condition} ===")
        missed = 0
        for item, text in zip(prepared, texts):
            verdict = parse_verdict(text)
            missed += verdict is None
            preview = " ".join(text.split())[:200]
            print(
                f"pair {item['pair_number']} target {item['target']} -> {verdict} "
                f"({len(item['input_ids'])} prompt tokens) | {preview}"
            )
        estimate = elapsed / len(prepared) * N_FUNCTIONS / 60
        print(f"{elapsed:.1f}s for {len(prepared)} functions. Rough estimate for all {N_FUNCTIONS}: {estimate:.0f} min.")
        if missed:
            print(
                f"{missed} reply(ies) did not parse to YES or NO. "
                "Read the text above before the full run. Unparsed pairs are not counted as benign."
            )


def run_all():
    if not RUN_CONDITIONS:
        raise RuntimeError("RUN_CONDITIONS is empty.")
    unknown = [name for name in RUN_CONDITIONS if name not in ("zero_shot", "two_shot", "cot")]
    if unknown:
        raise RuntimeError(f"Unknown conditions: {unknown}")
    warn_if_test_rows_match_examples()
    prepare_run_files()
    try:
        for condition in RUN_CONDITIONS:
            evaluate_condition(condition)
            gc.collect()
            torch.cuda.empty_cache()
    finally:
        summarize()


In [ ]:
# Read this output before the full run. A few minutes, not the full job.
smoke_test()


In [ ]:
# Full evaluation. Safe to rerun after a disconnect when RESUME is True.
run_all()


## Summary

The full-run cell already prints this table. Run this cell on its own after a reconnect: run the setup cell and the prompt cell again, then this cell. You can skip the model load when you only want the saved numbers.

For the check-in, use the four counts on each complete row. The UniXcoder line is the finished fine-tune (`8 / 76 / 476 / 4`), not a result from this notebook. Files:

- `MyDrive/primevul_qwen_icl/results/metrics.json`
- `MyDrive/primevul_qwen_icl/results/summary.csv`
- `MyDrive/primevul_qwen_icl/results/zero_shot.jsonl`, `two_shot.jsonl`, and `cot.jsonl`


In [ ]:
summarize()
